# Appendix: Coding Agents

A coding agent explores a repository, edits files, runs checks, and repeats until the requested outcome is verified.


## What to learn

- Repository search builds a focused view of unfamiliar code.
- Small patches preserve surrounding work and simplify review.
- Tests, type checks, linters, and diffs provide feedback.
- Sandbox and permission rules limit filesystem, network, and command access.

## Decision rule

Give the agent a clear outcome and relevant checks. Require it to inspect before editing, preserve unrelated changes, validate in proportion to risk, and report what remains unverified.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
"""The coding-agent inner loop: search/replace edits with pre-apply
verification, then an edit-test cycle that reads failures and retries.
"""

FILE = {"calc.py": "def total(items):\n    return sum(i.price for i in items)\n"}


# Define the data shape and small operations before running them.
def apply_search_replace(filename, search, replace):
    """The format's whole point: verifiable BEFORE side effects."""
    content = FILE[filename]
    if search not in content:
        return {"ok": False, "error": "search text not found - no changes made"}
    if content.count(search) > 1:
        return {"ok": False, "error": "search text ambiguous (2+ matches)"}
    FILE[filename] = content.replace(search, replace)
    return {"ok": True}


def run_tests():
    """Toy test: total() must skip items with price=None."""
    code = FILE["calc.py"]
    if "if i.price" in code or "is not None" in code:
        return {"passed": True}
    return {"passed": False,
            "output": "TypeError: unsupported operand +: int and NoneType (test_skips_none)"}


# The agent's scripted attempts: first a bad edit (stale quote), then correct.
ATTEMPTS = [
    {"search": "return sum(i.cost for i in items)",   # hallucinated old code
     "replace": "return sum(i.cost or 0 for i in items)"},
    {"search": "return sum(i.price for i in items)",
     "replace": "return sum(i.price for i in items if i.price is not None)"},
]


def edit_test_loop(max_cycles=3):
    for cycle, edit in enumerate(ATTEMPTS[:max_cycles]):
        result = apply_search_replace("calc.py", edit["search"], edit["replace"])
        if not result["ok"]:
            print(f"cycle {cycle}: edit REJECTED ({result['error']}) -> retry with fresh read")
            continue  # file untouched — the format prevented corruption
        tests = run_tests()
        print(f"cycle {cycle}: edit applied, tests "
              f"{'PASS' if tests['passed'] else 'FAIL: ' + tests['output']}")
        if tests["passed"]:
            return "green - ready for human diff review"
    return "budget exhausted - escalate with failing output"


print(edit_test_loop())
print("\nfinal file:\n" + FILE["calc.py"])


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
